# Exercise 2 - River Flow Velocity from Video

In this exercise, we will use short video captures of a high-altitude river to measure how fast the water is moving. We can do this via *pixel tracking*, which lets us see how different parts of an image change through time. 

Let's first load in some key modules for working with the video data.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import datetime

In [ ]:
import cv2 #This is the industry standard open source image manipulation library
video_file = '../VideoData/2024-02-10T06-00_rpi_chilime.h264' #This is the video we will work with

Let's first open the video file and look at the data one frame at a time:

In [ ]:
def show_image(img, title="Image", cmap=None):
    plt.figure(figsize=(10, 6))
    if len(img.shape) == 3:
        #Convert BGR (OpenCV standard) to RGB (Matplotlib standard)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.show()

cap = cv2.VideoCapture(video_file) #Read the video file
ret1, frame1 = cap.read() #Get the first frame
show_image(frame1, "Frame 1")

The idea is to take the frames and look at how they change -- let's compare two neighboring frames as well:

In [ ]:
def get_neighboring_frames(video_path):
    cap = cv2.VideoCapture(video_path)

    #Read two frames
    ret1, frame1 = cap.read()

    for i in range(100): #Read several frames to skip forward in time
        ret2, frame2 = cap.read()
    
    cap.release()
    
    return frame1, frame2

frame_a, frame_b = get_neighboring_frames(video_file)

if frame_a is not None:
    # Concatenate horizontally for side-by-side comparison
    combined = np.hstack((frame_a, frame_b))
    show_image(combined, "Left: Frame 1 | Right: Frame 2")

There is a LOT of information in those images -- often for computer vision problems it is easier to convert color images to greyscale. This reduces image complexity while still maintaining the most important information. We can quickly convert our frames and plot them again:

In [ ]:
def preprocess_frames(img1, img2):
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    return gray1, gray2

gray_a, gray_b = preprocess_frames(frame_a, frame_b)
show_image(np.hstack((gray_a, gray_b)), "Grayscale Frames", cmap='gray')

While the two images still look very similar, there are enough differences that computer vision algorithms can track how things have changed. There are many different ways to do this, but they generally rely on searching nearby areas for similar pixels through time. In this way the algorithm can 'match' where a certain feature (like a tree, or a rock, or a white patch in the water) has moved to. Let's try the simple Farneback algorithm, which is widely used for different contexts. 

You can find more information on the algorithm here: [https://docs.opencv.org/3.4/d4/dee/tutorial_optical_flow.html](https://docs.opencv.org/3.4/d4/dee/tutorial_optical_flow.html)

In [ ]:
def calculate_flow(prev, curr):
    #Calculate Dense Optical Flow
    #The result 'flow' is an array of shape (height, width, 2)
    #channel 0 = x movement (dx), channel 1 = y movement (dy)
    flow = cv2.calcOpticalFlowFarneback(prev, curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    return flow

def visualize_flow_quiver(img, flow, step=16, title="Raw Flow"):
    h, w = img.shape
    y, x = np.mgrid[step//2:h:step, step//2:w:step].reshape(2,-1).astype(int)
    
    fx, fy = flow[y, x].T
    
    plt.figure(figsize=(10, 6))
    plt.imshow(img, cmap='gray')
    
    #Quiver plots arrows. We invert Y because image origin is top-left
    plt.quiver(x, y, fx, -fy, color='cyan', scale_units='xy', angles='xy', scale=0.5)
    plt.title(title)
    plt.axis('off')
    plt.show()

raw_flow = calculate_flow(gray_a, gray_b)
visualize_flow_quiver(gray_a, raw_flow, step=16, title="Raw Optical Flow Vectors")

We can see that the main things that moved are the bushes to the side of the river! The river itself moved rather more slowly. 

Let's focus our analysis to a smaller region of the image -- this will make things faster if we want to process the whole video as well. 

In [ ]:
def subset_image(image):
    rotated_full = cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    image = image[:,800:-400]
    rotated = cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    return rotated, rotated_full

subset, full = subset_image(gray_a)
#show_image(full, "Full Image", cmap='gray')
show_image(subset, "Subset Image", cmap='gray')

That looks better! We also rotate the image at the same time to make this easier to see.

Let's re-run the movement analysis, and also run a filter to remove low-magnitude movement.

In [ ]:
def filter_low_magnitude(flow, threshold=1.0):
    #Calculate magnitude: sqrt(dx^2 + dy^2)
    mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    
    #Create a mask where magnitude is less than threshold
    mask = mag < threshold
    
    #Create a copy and set low-movement vectors to 0
    filtered_flow = flow.copy()
    filtered_flow[mask] = 0
    
    return filtered_flow

subset_a, _ = subset_image(gray_a)
subset_b, _ = subset_image(gray_b)
raw_flow = calculate_flow(subset_a, subset_b)
masked_flow = filter_low_magnitude(raw_flow, threshold=10.0)
visualize_flow_quiver(subset_a, masked_flow, step=8, title="Flow with Low Movement Masked")

Another way to filter the data is to check how consistent the movement is -- if most of the vectors point to the right and one points left, it is likely an error. Water in a river generally flows in similar directions!

In [ ]:
def filter_spatial_consistency(flow, step=16, angle_threshold_deg=45):
    h, w = flow.shape[:2]
    #We will only process the grid points we actually visualize to save time
    y_grid, x_grid = np.mgrid[step:h-step:step, step:w-step:step].reshape(2,-1).astype(int)
    
    consistent_flow = flow.copy()
    
    #Pre-calculate angles for the whole image
    _, angles = cv2.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=True)
    
    for y, x in zip(y_grid, x_grid):
        #Get current angle
        current_angle = angles[y, x]
        
        #Get angles of the 8 neighbors
        neighbors = angles[y-step:y+step+1:step, x-step:x+step+1:step]
        
        #Flatten and remove the center pixel (the one we are checking)
        neighbor_angles = neighbors.flatten()        
        median_angle = np.median(neighbor_angles)
        diff = abs(current_angle - median_angle)
        
        #Handle 360 degree wrap-around (e.g., 359 vs 1 is only 2 degrees diff)
        diff = min(diff, 360 - diff)
        
        if diff > angle_threshold_deg:
            consistent_flow[y, x] = [0, 0] #Remove inconsistent vector

    return consistent_flow

cleaned_flow = filter_spatial_consistency(masked_flow, step=16)
visualize_flow_quiver(subset_a, cleaned_flow, step=8, title="Spatially Filtered Flow")

The next step is to move beyond only looking at two frames -- we have a whole video to work with. We want to average the flow (speed and direction) over time to get a better idea of how the water is moving. The video captures 30 seconds, so the flow speed should be fairly smooth over that time period.

In [ ]:
#Count the number of frames to know how to average the data over the whole video

cap = cv2.VideoCapture(video_file)
def manual_count(handler):
    frames = 0
    while True:
        status, frame = handler.read()
        if not status:
            break
        frames += 1
    return frames 

manual_count(cap)

In [ ]:
def calculate_median_flow(video_path, max_frames=900, subsample_rate=5):
    def subset_image(image):
        image = image[:, 800:-400]
        rotated = cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
        return rotated

    cap = cv2.VideoCapture(video_path)

    ret, prev_frame = cap.read()
    if not ret: 
        return None, None
    
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    prev_gray = subset_image(prev_gray)

    flow_history = []
    frame_count = 0
    
    print(f"Processing up to {max_frames} frames (keeping 1 in every {subsample_rate})...")
    
    while True:
        if frame_count >= max_frames:
            break
            
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = subset_image(gray)
        
        #Calculate flow for the current step
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        #Only append to history if we hit the subsample rate
        if frame_count % subsample_rate == 0:
            # Cast to float32 to save 50% memory compared to float64
            flow_history.append(flow.astype(np.float32))
            
        frame_count += 1
        prev_gray = gray 

        if frame_count % 100 == 0:
            print(f"Processed {frame_count} frames...")
        
    cap.release()
    print(f"\nFinished. Computing Median of {len(flow_history)} stored frames...")
    
    #Stack list into a 4D array (Time, Height, Width, 2)
    stack = np.stack(flow_history, axis=0)
    
    #Calculate Median along the time axis (axis 0)
    median_flow = np.median(stack, axis=0)
    
    return median_flow, prev_gray

median_flow_vectors, last_frame = calculate_median_flow(video_file, max_frames=250)
final_result = filter_spatial_consistency(median_flow_vectors, step=8)
visualize_flow_quiver(last_frame, final_result, step=8, title="Median Flow (Time Aggregated)")

We can make the visual a little clearer by scaling the vectors and coloring them -- this makes it easier to see the fast/slow areas:

In [ ]:
def visualize_enhanced_flow(img, flow, step=8, scale=0.1, max_speed=100):
    h, w = img.shape
    y, x = np.mgrid[step//2:h:step, step//2:w:step].reshape(2,-1).astype(int)
    
    #Get vectors
    fx, fy = flow[y, x].T
    
    #Calculate speed (magnitude) for coloring
    magnitude = np.sqrt(fx**2 + fy**2)
    magnitude[magnitude >= max_speed] = np.nan
    
    plt.figure(figsize=(12, 8))
    
    #Display the background image
    plt.imshow(img, cmap='gray', alpha=0.8)
    
    #Quiver Plot Settings:
    #x, y, u, v: positions and vectors
    #magnitude: this is the array used to pick colors
    #cmap: 'plasma' is great for flow (Blue=Slow, Yellow=Fast)
    #scale: The LOWER this number, the LONGER the arrows.
    #width: Thickness of the arrow shaft
    quiver = plt.quiver(x, y, fx, -fy, magnitude, 
                        cmap='plasma', 
                        scale=scale, 
                        scale_units='inches',
                        width=0.003, 
                        headwidth=2,
                        minshaft=1) # Enforce a minimum length
    
    #Add a color bar to explain speeds
    cbar = plt.colorbar(quiver)
    cbar.set_label('Water Speed (pixels/frame)')
    
    plt.title("Median River Velocity")
    plt.axis('off')
    plt.show()

visualize_enhanced_flow(last_frame, final_result, step=8, scale=2)

We can also focus in on the slower parts of the river by masking out fast regions:

In [ ]:
visualize_enhanced_flow(last_frame, final_result, step=8, scale=0.5, max_speed=0.25)

## Conclusions

This was a very basic introduction to what we can do with image processing tools -- there are many other applications for pixel tracking across the geosciences. There are a wide range of algorithms, and many different ways to pre-process different kinds of data.

In the next demonsration, we will go one step further and implement a *block matching* algorithm. This is a more computationally expensive method, but generally provides better results. The main difference is that with the Farneback algorithm (Dense Optical Flow, used above) we rely on polynomials and gradients to compute flow, whereas with block matching (below), we look at correlation. Farneback is much faster, but can be unreliable in some contexts. 